# Notebook 1: PVS Dataset Preparation & Central IMU Merging

This notebook extracts sensor readings from the multi-class PVS dataset (`PVS 1` through `PVS 9`). In physical test vehicles, smartphones were mounted on both the left and right sides of the dashboard. To obtain a unified chassis response representing the sprung mass center, we compute the arithmetic mean of the left and right accelerometer and gyroscope axes:
$$a_{x,\text{mid}} = \frac{a_{x,\text{left}} + a_{x,\text{right}}}{2}, \quad a_{y,\text{mid}} = \frac{a_{y,\text{left}} + a_{y,\text{right}}}{2}, \quad a_{z,\text{mid}} = \frac{a_{z,\text{left}} + a_{z,\text{right}}}{2}$$
$$\omega_{x,\text{mid}} = \frac{\omega_{x,\text{left}} + \omega_{x,\text{right}}}{2}, \quad \omega_{y,\text{mid}} = \frac{\omega_{y,\text{left}} + \omega_{y,\text{right}}}{2}, \quad \omega_{z,\text{mid}} = \frac{\omega_{z,\text{left}} + \omega_{z,\text{right}}}{2}$$
This averaging physically eliminates roll-induced differential vertical vibrations across opposite sides of the vehicle, isolating pure heave and pitch dynamics.

Furthermore, because physical gyroscope sensors in the PVS dataset log angular velocity in **degrees/sec**, whereas our Section 7 model is trained on SI units (**radians/sec**), we scale all gyroscope readings during extraction:
$$\omega_{\text{rad/s}} = \omega_{\text{deg/s}} \times \frac{\pi}{180}$$

We also fuse the categorical road quality labels from `dataset_labels.csv`. Road conditions are encoded as integer classes:
- **Good Road (`0`)**: Smooth asphalt or paved road without severe defects.
- **Regular Road (`1`)**: Moderate roughness or minor degradation.
- **Bad Road (`2`)**: Severe roughness, cobblestones, dirt roads, or structural deterioration.

To combine the binary one-hot left and right labels into a single ground truth quality label, we use the logical OR / maximum severity:
$$\text{label}_{\text{mid}} = \max(\text{label}_{\text{left}}, \text{label}_{\text{right}})$$


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- CONFIGURATION ---
RAW_DATA_DIR = r"D:\Coding\Hackathon\GFG\ARM\ARM\ml_model_work\data\raw\multi_class_kaggle"
OUTPUT_DATA_DIR = r"D:\Coding\Hackathon\GFG\ARM\ARM\ml_model\review_work\data"
OUTPUT_FIG_DIR = r"D:\Coding\Hackathon\GFG\ARM\ARM\ml_model\review_work\outputs\figures"

os.makedirs(OUTPUT_DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_FIG_DIR, exist_ok=True)
print(f"[*] Raw Data Directory: {RAW_DATA_DIR}")
print(f"[*] Output Data Directory: {OUTPUT_DATA_DIR}")


In [ ]:
def process_pvs_trip(trip_id):
    folder_path = os.path.join(RAW_DATA_DIR, f"PVS {trip_id}")
    left_csv = os.path.join(folder_path, "dataset_gps_mpu_left.csv")
    right_csv = os.path.join(folder_path, "dataset_gps_mpu_right.csv")
    labels_csv = os.path.join(folder_path, "dataset_labels.csv")
    
    if not (os.path.exists(left_csv) and os.path.exists(right_csv) and os.path.exists(labels_csv)):
        print(f"[!] Skipping PVS {trip_id}: Missing required CSV files in {folder_path}")
        return None
        
    print(f"[+] Processing PVS {trip_id}...")
    
    # Load raw CSVs
    df_l = pd.read_csv(left_csv)
    df_r = pd.read_csv(right_csv)
    df_lab = pd.read_csv(labels_csv)
    
    # Verify row alignment
    min_len = min(len(df_l), len(df_r), len(df_lab))
    if len(df_l) != min_len or len(df_r) != min_len or len(df_lab) != min_len:
        print(f"    [!] Length mismatch in PVS {trip_id} (Left: {len(df_l)}, Right: {len(df_r)}, Labels: {len(df_lab)}). Truncating to min_len={min_len}.")
        df_l = df_l.iloc[:min_len].reset_index(drop=True)
        df_r = df_r.iloc[:min_len].reset_index(drop=True)
        df_lab = df_lab.iloc[:min_len].reset_index(drop=True)
        
    # Create central DataFrame
    df_mid = pd.DataFrame()
    
    # 1. Timestamps and Odometry (Primary reference: left sensor)
    df_mid['timestamp'] = df_l['timestamp']
    df_mid['timestamp_gps'] = df_l['timestamp_gps']
    df_mid['latitude'] = df_l['latitude']
    df_mid['longitude'] = df_l['longitude']
    df_mid['speed'] = df_l['speed']  # speed in m/s
    
    # 2. Central IMU Averaging (Sprung Mass Center)
    imu_axes = ['acc_x_dashboard', 'acc_y_dashboard', 'acc_z_dashboard', 
                'gyro_x_dashboard', 'gyro_y_dashboard', 'gyro_z_dashboard']
    
    for col in imu_axes:
        short_name = col.replace('_dashboard', '').replace('acc_', 'a').replace('gyro_', 'w')
        val = (df_l[col].astype(np.float32) + df_r[col].astype(np.float32)) / 2.0
        if 'gyro_' in col:
            val = val * (np.pi / 180.0)  # Scale angular velocity from deg/sec to rad/sec
        df_mid[short_name] = val
        
    # 3. Ground Truth Road Quality Categorical Labels (0=Good, 1=Regular, 2=Bad)
    label_left = np.zeros(min_len, dtype=np.int32)
    label_left[df_lab['regular_road_left'] == 1] = 1
    label_left[df_lab['bad_road_left'] == 1] = 2
    
    label_right = np.zeros(min_len, dtype=np.int32)
    label_right[df_lab['regular_road_right'] == 1] = 1
    label_right[df_lab['bad_road_right'] == 1] = 2
    
    # Logical OR / Maximum Worst-Case Severity
    df_mid['label_left'] = label_left
    df_mid['label_right'] = label_right
    df_mid['label_mid'] = np.maximum(label_left, label_right)
    
    # 4. Surface Metadata and Anomalies
    surface_cols = [
        'paved_road', 'unpaved_road', 'dirt_road', 'cobblestone_road', 
        'asphalt_road', 'no_speed_bump', 'speed_bump_asphalt', 'speed_bump_cobblestone'
    ]
    for col in surface_cols:
        if col in df_lab.columns:
            df_mid[col] = df_lab[col].astype(np.int32)
        else:
            df_mid[col] = 0
            
    # Save processed central dataset
    out_path = os.path.join(OUTPUT_DATA_DIR, f"PVS_{trip_id}_central.csv")
    df_mid.to_csv(out_path, index=False)
    
    label_counts = df_mid['label_mid'].value_counts().to_dict()
    print(f"    [✔] Saved {len(df_mid):,} samples to {out_path}")
    print(f"        Class Distribution -> Good (0): {label_counts.get(0, 0):,} | Regular (1): {label_counts.get(1, 0):,} | Bad (2): {label_counts.get(2, 0):,}")
    
    return df_mid


In [ ]:
# --- PROCESS ALL 9 TRIPS ---
trip_summaries = []
all_mid_labels = []

for i in range(1, 10):
    df_trip = process_pvs_trip(i)
    if df_trip is not None:
        trip_summaries.append({
            'trip_id': f"PVS {i}",
            'total_samples': len(df_trip),
            'good_samples': (df_trip['label_mid'] == 0).sum(),
            'regular_samples': (df_trip['label_mid'] == 1).sum(),
            'bad_samples': (df_trip['label_mid'] == 2).sum(),
            'duration_sec': round(df_trip['timestamp'].max() - df_trip['timestamp'].min(), 1),
            'distance_km': round((df_trip['speed'] * df_trip['timestamp'].diff().fillna(0.01)).sum() / 1000.0, 2)
        })
        all_mid_labels.extend(df_trip['label_mid'].values)

summary_df = pd.DataFrame(trip_summaries)
print("\n" + "="*80)
print("PVS DATASET CENTRAL IMU EXTRACTION SUMMARY")
print("="*80)
display(summary_df)

summary_csv_path = os.path.join(OUTPUT_DATA_DIR, "pvs_extraction_summary.csv")
summary_df.to_csv(summary_csv_path, index=False)
print(f"\n[✔] Exported summary table to: {summary_csv_path}")


In [ ]:
# --- VISUALIZE CLASS DISTRIBUTION & CENTRAL AVERAGING ---
plt.figure(figsize=(14, 5))

# Subplot 1: Class Distribution
plt.subplot(1, 2, 1)
counts = pd.Series(all_mid_labels).value_counts().sort_index()
bars = plt.bar(['Good (0)', 'Regular (1)', 'Bad (2)'], counts.values, color=['#2ca02c', '#ff7f0e', '#d62728'], alpha=0.85, edgecolor='black')
plt.title("Combined PVS Ground Truth Quality Distribution (label_mid)", fontsize=12, fontweight='bold')
plt.xlabel("Road Quality Class", fontsize=11)
plt.ylabel("Number of Sample Points", fontsize=11)
plt.grid(axis='y', linestyle='--', alpha=0.5)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + (max(counts.values)*0.01), f"{int(yval):,}", ha='center', va='bottom', fontweight='bold')

# Subplot 2: Demonstration of Roll Cancellation (10-second slice from PVS 1)
try:
    sample_trip_l = pd.read_csv(os.path.join(RAW_DATA_DIR, "PVS 1", "dataset_gps_mpu_left.csv"), nrows=1000)
    sample_trip_r = pd.read_csv(os.path.join(RAW_DATA_DIR, "PVS 1", "dataset_gps_mpu_right.csv"), nrows=1000)
    
    t = sample_trip_l['timestamp'] - sample_trip_l['timestamp'].iloc[0]
    az_l = sample_trip_l['acc_z_dashboard']
    az_r = sample_trip_r['acc_z_dashboard']
    az_mid = (az_l + az_r) / 2.0
    
    plt.subplot(1, 2, 2)
    plt.plot(t[:300], az_l[:300], label='Left Dashboard az', color='#1f77b4', alpha=0.4, linewidth=1)
    plt.plot(t[:300], az_r[:300], label='Right Dashboard az', color='#aec7e8', alpha=0.4, linewidth=1)
    plt.plot(t[:300], az_mid[:300], label='Central Averaged az (Sprung Mass)', color='#d62728', linewidth=2)
    plt.title("Central IMU Averaging (Roll Vibration Cancellation)", fontsize=12, fontweight='bold')
    plt.xlabel("Time (seconds)", fontsize=11)
    plt.ylabel("Vertical Acceleration (m/s²)", fontsize=11)
    plt.legend(loc='upper right', frameon=True)
    plt.grid(True, linestyle='--', alpha=0.5)
except Exception as e:
    print(f"[!] Could not plot sample time-series: {e}")

plt.tight_layout()
plot_path = os.path.join(OUTPUT_FIG_DIR, "pvs_data_preparation_overview.png")
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"[✔] Overview visualization saved to: {plot_path}")
